<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Change points

**[Change points](https://learning.nextia-ai.com/courses/anomaly-detection/m05/change-points/)** · Anomaly Detection: Find What Is Unusual · Module 5, lesson 4 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the moments when a series moves to a new level and stays there, with PELT and binary segmentation (ruptures), compare them with a CUSUM chart, and treat a change point as a new normal, not as an alert.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 10 MB). No account. |
| **You should know** | STL, from [STL residuals](https://learning.nextia-ai.com/courses/anomaly-detection/m05/stl-residuals/); CUSUM, from [Small, lasting shifts: CUSUM and EWMA](https://learning.nextia-ai.com/courses/anomaly-detection/m03/small-lasting-shifts-cusum-and-ewma/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m05/change-points/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M05-L04-change-points/change-points-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2, statsmodels 0.15.0, ruptures 1.1.10. It needs an internet connection. The first time, it takes up to a minute. On Python 3.14 or newer, pip builds ruptures from its source code (ruptures 1.1.10 says it needs an older Python, but it works on 3.14). This needs a C compiler: Colab and Kaggle have one; on a Mac, run `xcode-select --install` once.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
import sys
flag = "--ignore-requires-python" if sys.version_info >= (3, 14) else ""
%pip install -q --disable-pip-version-check {flag} pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 statsmodels==0.15.0 ruptures==1.1.10
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib', 'statsmodels', 'ruptures']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`ticket_api_metrics.csv.gz`, `ticket_api_events.csv`, `payments.csv.gz`) into a folder `anomaly-course/data`, and checks each file's checksum. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "ticket_api_metrics.csv.gz": "73b420194586f72b760fd672f89ee2526035385965668a3098cf1e34dfa123c4",
    "ticket_api_events.csv": "3396099abee2b4326a301806fd7885b28f02e287ec1c455c6599bd280684d0c6",
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
}
CODE_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Where does the level change?

A road has a speed limit of 50, then a sign, then 80. The cars' speeds go up at the sign, and they stay up. The sign is a **change point**: a moment after which the normal level is different.

Run the next cell. It tries every place to cut 12 values into two parts. For each cut, the **cost** adds up how far each value is from the mean of its own part (squared). The best cut has the lowest cost.

> **Check.** The lowest cost is 9.33, for the cut before value 7. With no cut, the cost is 74.67.

In [ ]:
# Twelve values. Where would you cut them into two parts with a steady level each?
values = np.array([10, 11, 9, 10, 12, 10, 15, 14, 16, 15, 14, 16], dtype=float)
def cost(part):                                   # how far the values are from their own mean (squared)
    return ((part - part.mean()) ** 2).sum()
rows = [{"cut before value": k + 1, "left mean": values[:k].mean().round(2), "right mean": values[k:].mean().round(2),
         "cost": round(cost(values[:k]) + cost(values[k:]), 2)} for k in range(2, len(values) - 1)]
print("no cut: cost", round(cost(values), 2))
print(pd.DataFrame(rows).to_string(index=False))

A **change-point method** does this on a long series, and it can cut more than once. Each extra cut makes the cost smaller, so a method adds a **penalty** for every cut: it keeps a cut only if it lowers the cost by more than the penalty. **PELT** (Pruned Exact Linear Time) finds the best set of cuts for a penalty, fast. **Binary segmentation** cuts once, then cuts each part again, until it has the number of cuts you ask for. Both are in the `ruptures` package.

## 2. The ticket API's latency

The next cell loads 60 days of the ticket API's latency, hour by hour, and the events that are not weekly releases.

In [ ]:
import ruptures as rpt

metrics = pd.read_csv("data/ticket_api_metrics.csv.gz", parse_dates=["ts"]).set_index("ts")
events = pd.read_csv("data/ticket_api_events.csv", parse_dates=["ts"])
hourly = metrics.latency_p95_ms.resample("1h").mean()              # 1,440 hours
signal = np.log(hourly.to_numpy()).reshape(-1, 1)                 # the log: a rise of 10% is the same size everywhere
print(len(hourly), "hours")
print(events[events.kind != "release"].to_string(index=False))

> **Predict.** We planted a slow memory leak (2025-09-22 03:00 until the restart on 2025-09-27 04:30) and a configuration change (2025-10-20 10:00). Where will PELT cut?

Run the next cell. It runs PELT with a penalty of 2 × log(hours) × variance, and binary segmentation with 6 cuts.

In [ ]:
penalty = 2 * np.log(len(signal)) * np.var(signal)                # the price of one more change point
pelt = rpt.Pelt(model="l2", min_size=12).fit(signal).predict(pen=penalty)
binseg = rpt.Binseg(model="l2", min_size=12).fit(signal).predict(n_bkps=6)
print("PELT:               ", [str(hourly.index[i]) for i in pelt[:-1]])
print("binary segmentation:", [str(hourly.index[i]) for i in binseg[:-1]])
start = 0
for end in pelt:                                                  # the mean latency of each segment
    print(f"  {hourly.index[start]} to {hourly.index[end - 1]}: {np.exp(signal[start:end].mean()):.1f} ms")
    start = end

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(hourly.index, hourly, lw=0.6, color="grey", label="p95 latency, hourly mean (ms)")
for i in pelt[:-1]:
    ax.axvline(hourly.index[i], color="red", ls="--")
ax.axvspan(pd.Timestamp("2025-09-22 03:00"), pd.Timestamp("2025-09-27 04:30"), color="orange", alpha=0.2, label="planted leak")
ax.legend(loc="upper right")
plt.show()

PELT cuts 4 times: three cuts inside the leak and one cut at the configuration change. A slow rise is not one step, so PELT draws it as a staircase: each step is a new level. Binary segmentation puts two more steps in the same rise. Both methods look at the **whole** series, so they say what changed in the past; they are not live alerts.

## 3. CUSUM sees the change, then alarms again and again

CUSUM, from Module 3, adds up small deviations above normal and alarms when the sum is too large. Run the next cell: a CUSUM on the hourly latency, with the daily pattern removed, and "normal" learned on the first 14 days.

In [ ]:
from statsmodels.tsa.seasonal import STL
deseasoned = hourly - STL(hourly, period=24, robust=True).fit().seasonal   # remove the daily pattern
x = np.log(deseasoned.clip(lower=1).to_numpy())
base = x[: 14 * 24]                                                # the first 14 days are "normal"
z = (x - base.mean()) / base.std()
c, cusum_alarms = 0.0, []
for i, v in enumerate(z):                                          # CUSUM, as in Module 3: k = 0.5, h = 5
    c = max(0.0, c + v - 0.5)
    if c > 5:
        cusum_alarms.append(hourly.index[i])
        c = 0.0                                                    # start again after each alarm
cusum_alarms = pd.DatetimeIndex(cusum_alarms)
print("alarms before the leak:", int((cusum_alarms < "2025-09-22 03:00").sum()))
print("first alarm after the leak starts:", cusum_alarms[cusum_alarms >= "2025-09-22 03:00"].min())
print("alarms in total:", len(cusum_alarms), "| after the configuration change:", int((cusum_alarms >= "2025-10-20 10:00").sum()))

CUSUM finds the leak 6 hours after it starts: much faster than STL. It also had 6 false alarms in the three weeks before. And it alarms 235 times in total. After the configuration change, the latency stays about 12% higher, and CUSUM still compares it with the old normal: it alarms every few hours, for ten days. **A change point is a new normal.** After a lasting change, decide if it is a problem. If it is, fix it. If it is not (a new carrier, a new app), learn the new normal, or the detector alarms forever.

## 4. The product launch: a new normal in the payments

On 2025-10-15, Larkfield launched its app. Many customers started to pay from a phone that the shop had never seen for them. Run the next cell: the daily share of payments from a device that is new for the customer.

In [ ]:
payments = pd.read_csv("data/payments.csv.gz", usecols=["ts", "customer_id", "device_id"], parse_dates=["ts"])
payments["day"] = payments.ts.dt.normalize()
payments = payments.sort_values("ts")
first_day = payments.groupby("customer_id").day.transform("min")
first_seen = payments.groupby(["customer_id", "device_id"]).day.transform("min")
payments["new_device"] = (first_seen == payments.day) & (first_day < payments.day)   # a device never seen before
share = 100 * payments.groupby("day").new_device.mean()
share = share[share.index >= "2025-07-01"]
print(share.loc["2025-10-12":"2025-10-18"].round(2).to_string())

In [ ]:
app = share.to_numpy().reshape(-1, 1)
app_penalty = 2 * np.log(len(app)) * np.var(app[: (share.index < "2025-09-01").sum()])   # the noise of July and August
app_points = rpt.Pelt(model="l2", min_size=7).fit(app).predict(pen=app_penalty)
start = 0
for end in app_points:
    print(f"{share.index[start].date()} to {share.index[end - 1].date()}: {app[start:end].mean():.2f}% new device")
    start = end

PELT finds the start of the change near the launch, then more steps down: customers use the app again, and the device is no longer new for them. So "normal" moved, and then it moved again. This is why the detection job of Module 9 checks for drift and retrains.

## 5. Check yourself

**Your turn 1.** Run PELT on the hourly latency again with a penalty **4 times** larger (`4 * penalty`). List its change points as strings like `"2025-09-22 06:00:00"` in `strict_points`.

In [ ]:
strict_points = ...   # your code

In [ ]:
expect_hash('strict_points', strict_points, '4607e8624b853956')

**Your turn 2.** On which day does the first change point of the app-launch series start? Write it as a string `"YYYY-MM-DD"` in `first_app_change`. (A change point index is the first value of the new segment.)

In [ ]:
first_app_change = ...   # your code

In [ ]:
expect_hash('first_app_change', first_app_change, '415d013e692c2d7b')

With a larger penalty, PELT keeps 3 cuts: a larger penalty means fewer, larger changes. The penalty is a choice, like a threshold.

## Recap

- A change point is a moment after which the normal level is different. PELT and binary segmentation find them in a finished series; the penalty decides how many.
- A slow rise becomes a staircase of change points. A sudden step becomes one.
- CUSUM finds a change fast, but it keeps alarming until "normal" is learned again.
- After a change point, decide: a problem to fix, or a new normal to learn.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Many metrics at once](https://learning.nextia-ai.com/courses/anomaly-detection/m05/many-metrics-at-once/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m05/change-points/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.